# Unit 4 Exercise: The Decorator Pattern in a Coffee Shop

Ian Stevens, Advanced Object-Oriented Design and Programming, Unit 4.

The seminar asks for a coffee shop application in which toppings such as milk and sugar are added to a base coffee object at run time, using the Decorator pattern to enhance an existing class without changing it. The design is in `coffee.py` and its tests are in `test_coffee.py`. This notebook runs the design, compares it with the alternatives and records where the pattern stops helping.

In [1]:
import inspect
import itertools
import math
import platform

import coffee
from coffee import (BASES, TOPPINGS, Beverage, Coffee, Espresso, LoyaltyDiscount, Milk,
                    OrderError, Sugar, Topping, VanillaSyrup, audited, format_price, make_drink)

print("Python", platform.python_version())

Python 3.13.18


## 1. The four roles

Sarcar (2022, ch. 9) quotes the Gang of Four definition: the pattern 'attaches additional responsibilities to an object dynamically' and offers 'a flexible alternative to subclassing for extending functionality'. It has four roles. `Beverage` is the component, `Coffee` and `Espresso` are concrete components, `Topping` is the decorator, which is itself a `Beverage` and holds the drink it wraps, and the toppings are concrete decorators.

In [2]:
for cls in (Beverage, Coffee, Espresso, Topping, *TOPPINGS.values(), LoyaltyDiscount):
    extra = f"adds {cls.name!r} for {format_price(cls.price)}" if hasattr(cls, "name") else ""
    print(f"{cls.__name__:<16} parent={cls.__mro__[1].__name__:<9} {extra}")

for attempt in (lambda: Beverage(), lambda: Topping(Coffee())):
    try:
        attempt()
    except TypeError as error:
        print("TypeError:", error)

Beverage         parent=ABC       
Coffee           parent=Beverage  
Espresso         parent=Beverage  
Topping          parent=Beverage  
Milk             parent=Topping   adds 'milk' for £0.40
OatMilk          parent=Topping   adds 'oat milk' for £0.55
Sugar            parent=Topping   adds 'sugar' for £0.15
VanillaSyrup     parent=Topping   adds 'vanilla syrup' for £0.45
ExtraShot        parent=Topping   adds 'extra shot' for £0.60
WhippedCream     parent=Topping   adds 'whipped cream' for £0.50
LoyaltyDiscount  parent=Topping   adds 'loyalty discount' for £0.00
TypeError: Can't instantiate abstract class Beverage without an implementation for abstract methods 'cost', 'description'
TypeError: Topping is abstract; use a concrete topping


`Topping` has no abstract method left, because it forwards both of them, so Python would happily instantiate it. Sarcar's `Luxury` decorator has the same shape and Java's `abstract` keyword stops it being created (2022, ch. 9, Q&A 9.6). Python has no equivalent for a class with no abstract methods, so `Topping.__init__` checks by hand.

## 2. The demonstration

Toppings wrap the coffee one layer at a time, and each layer forwards the call inward and adds its own part on the way out.

In [3]:
drink = Coffee()
print(drink)
drink = Milk(drink)
print(drink)
drink = Sugar(drink)
print(drink)
print("layers, innermost first:", drink.layers())
print()
coffee.main();

<Coffee: Coffee (regular): £2.50>
<Milk: Coffee (regular), milk: £2.90>
<Sugar: Coffee (regular), milk, sugar: £3.05>
layers, innermost first: ('Coffee (regular)', 'milk', 'sugar')

Coffee (regular): £2.50
Coffee (regular), milk, sugar: £3.05
Coffee (large), oat milk, vanilla syrup, sugar, sugar: £4.10
Espresso, extra shot, whipped cream: £3.20


## 3. Why not subclasses?

Without the pattern, every combination the shop sells needs its own class (`CoffeeWithMilkAndSugar` and so on). With two base classes and six toppings, and allowing each topping at most once, that is one class per base for every subset of toppings. The decorator design needs one class per base and one per topping.

In [4]:
bases, toppings = 2, len(TOPPINGS)  # Coffee and Espresso
print("subclass per combination:", bases * 2 ** toppings)
print("decorator design:        ", 2 + 1 + toppings, "(Coffee, Espresso, Topping and one per topping)")
print("adding one more topping doubles the first and adds one to the second")

subclass per combination: 128
decorator design:         9 (Coffee, Espresso, Topping and one per topping)
adding one more topping doubles the first and adds one to the second


The count is conservative. It treats size as an attribute rather than a class and ignores double shots and second sugars, which would need still more classes.

## 4. The order desk and untrusted input

A drink can be assembled by hand, but orders arrive as text. `make_drink()` turns names into toppings through read-only tables, so a name can only ever select something on the menu. It never looks a name up as an attribute or class, which matters when the text comes from a web form or an app. It also enforces limits that no single topping can see, because each layer knows only the drink directly inside it.

In [5]:
print(make_drink("Large Coffee", ["oat milk", "vanilla syrup"]).receipt_line())

for base, extras in [("coffee", ["ketchup"]), ("coffee", ["__class__"]),
                     ("coffee", ["sugar"] * 4), ("tea", [])]:
    try:
        make_drink(base, extras)
    except OrderError as error:
        print("OrderError:", error)

try:
    TOPPINGS["free shot"] = coffee.ExtraShot
except TypeError as error:
    print("TypeError:", error)

Coffee (large), oat milk, vanilla syrup: £3.80
OrderError: not on the menu: 'ketchup'
OrderError: not on the menu: '__class__'
OrderError: at most 3 of each topping: sugar
OrderError: 'tea' is not on the menu
TypeError: 'mappingproxy' object does not support item assignment


## 5. Python's `@decorator` is a different thing

Ayeva and Kasampalis (2024, ch. 4) point out that there is no one-to-one relationship between the pattern and Python's decorator syntax. Francesco (2024, ch. 2) makes a similar move in JavaScript, replacing his `InstrumentedHttpClient` class with an `addTiming` function. `audited()` is the function version: it wraps `make_drink` once, when the program is assembled, rather than wrapping one drink at a time. `functools.wraps` keeps the original function's name and docstring.

In [6]:
log = []
desk = audited(log)(make_drink)
desk("espresso", ["extra shot"])
try:
    desk("coffee", ["ketchup"])
except OrderError:
    pass
print(desk.__name__, "|", desk.__doc__.splitlines()[0])
print(*log, sep="\n")

make_drink | Build a drink from the names on an order.
make_drink -> <ExtraShot: Espresso, extra shot: £2.70>
make_drink refused: not on the menu: 'ketchup'


## 6. Where the pattern stops helping

Three limits came out of the tests. First, a decorated drink is a `Beverage` but no longer an `Espresso`, so any code that checks the concrete class will treat it differently.

In [7]:
drink = Milk(Espresso())
print("isinstance Beverage:", isinstance(drink, Beverage))
print("isinstance Espresso:", isinstance(drink, Espresso))

isinstance Beverage: True
isinstance Espresso: False


Second, Francesco (2024, ch. 2) describes decorators as adding 'new members' to the object, such as the `requestTimings` field on his `InstrumentedHttpClient`, and says that this is why decorators can stack. Adding a member and stacking pull against each other: the next layer does not forward the new member, so only the outermost layer's additions are reachable.

In [8]:
class Labelled(Topping):
    name, price = "name on cup", 0
    def __init__(self, drink, label):
        super().__init__(drink)
        self.label = label

inner = Labelled(Coffee(), "Ian")
print("label on the inner layer:", inner.label)
try:
    Sugar(inner).label
except AttributeError as error:
    print("AttributeError:", error)

label on the inner layer: Ian
AttributeError: 'Sugar' object has no attribute 'label'


Third, toppings that only add a price can go in any order, but a decorator that alters the result cannot. A loyalty discount applied inside the syrup does not discount the syrup. `make_drink()` always puts the discount outermost, which is a rule the decorators themselves cannot enforce.

In [9]:
for drink in (LoyaltyDiscount(VanillaSyrup(Coffee())), VanillaSyrup(LoyaltyDiscount(Coffee()))):
    print(f"{' > '.join(drink.layers()):<46} {format_price(drink.cost())}")

orders = set()
for order in itertools.permutations([Milk, Sugar, VanillaSyrup]):
    d = Espresso()
    for t in order:
        d = t(d)
    orders.add(d.cost())
print("additive toppings in all 6 orders give", len(orders), "price:", format_price(orders.pop()))

Coffee (regular) > vanilla syrup > loyalty discount £2.66
Coffee (regular) > loyalty discount > vanilla syrup £2.70
additive toppings in all 6 orders give 1 price: £3.10


## 7. Running the tests

In [10]:
import io
import unittest
import test_coffee

def flatten(suite):
    for item in suite:
        yield from flatten(item) if isinstance(item, unittest.TestSuite) else [item]

suite = unittest.defaultTestLoader.loadTestsFromModule(test_coffee)
names = [test.id().replace("test_coffee.", "") for test in flatten(suite)]
result = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(suite)
print(f"Ran {result.testsRun} tests: {len(result.failures)} failures, {len(result.errors)} errors")
print(*("  " + name for name in names), sep="\n")

Ran 27 tests: 0 failures, 0 errors
  BriefRequirements.test_base_coffee_has_a_description_and_a_price
  BriefRequirements.test_base_coffee_is_unchanged_by_wrapping
  BriefRequirements.test_coffee_class_knows_nothing_about_toppings
  BriefRequirements.test_demonstration_prints_receipts
  BriefRequirements.test_milk_and_sugar_wrap_a_coffee
  BriefRequirements.test_toppings_are_added_at_run_time_from_data
  Limits.test_a_decorator_that_alters_the_price_makes_order_matter
  Limits.test_decorated_drink_is_no_longer_an_instance_of_its_base
  Limits.test_members_added_by_a_decorator_are_hidden_by_the_next_layer
  Limits.test_order_desk_puts_the_discount_outermost
  OrderDesk.test_a_new_topping_can_be_offered_without_editing_coffee_py
  OrderDesk.test_builds_the_drink_named_on_the_order
  OrderDesk.test_function_decorator_audits_orders_and_keeps_the_name
  OrderDesk.test_menu_tables_are_read_only
  OrderDesk.test_names_cannot_reach_attributes_or_other_classes
  OrderDesk.test_shop_limits_are_e

## References

Ayeva, K. and Kasampalis, S. (2024) *Mastering Python design patterns: craft essential Python patterns by following core design principles*. 3rd edn. Birmingham: Packt Publishing.

Francesco, H.D. (2024) *JavaScript design patterns: deliver fast and efficient production-grade JavaScript applications at scale*. Birmingham: Packt Publishing.

Sarcar, V. (2022) *Java design patterns: a hands-on experience with real-world examples*. 3rd edn. Berkeley, CA: Apress.